# Market Impact Decay: Buy-Sell Combined Analysis

Combined market impact = (buy_return - sell_return) / 2  
This removes directional bias and isolates the symmetric market impact signal.

**Key question:** Does the model reproduce market impact decay (power-law price reversion after aggressive orders stop)? How does generation horizon affect this?

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objs as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
from scipy.optimize import curve_fit
import re

## Configuration

In [ ]:
AGGRESSIVE_ORDER_ID = 77777777
MAX_SAMPLES = 500          # max samples to load per folder
MIDPRICE_MAX = 2_000_000   # filter samples where midprice exceeds this (outliers)

CONFIGS = {
    500: {
        'buy_path': Path("/app/output/evalsequences/aggressive_scenario/context_500_buy"),
        'sell_path': Path("/app/output/evalsequences/aggressive_scenario/context_500_sell"),
        'folders': [
            "i3_c12_mb5_cntxt15%",
            "i3_c12_mb15_cntxt45%",
            "i3_c12_mb25_cntxt75%",
            "i3_c12_mb50_cntxt150%",
            "i5_c15_mb5_cntxt20%",
            "i5_c15_mb15_cntxt60%",
            "i5_c15_mb25_cntxt100%",
            "i5_c15_mb50_cntxt200%",
        ],
    },
    250: {
        'buy_path': Path("/app/output/evalsequences/aggressive_scenario/context_250_buy"),
        'sell_path': Path("/app/output/evalsequences/aggressive_scenario/context_250_sell"),
        'folders': [
            "i3_c30_mb5_cntxt66%",
            "i3_c30_mb15_cntxt198%",
            "i3_c30_mb25_cntxt330%",
            "i3_c30_mb50_cntxt660%",
            "i5_c50_mb5_cntxt110%",
            "i5_c50_mb15_cntxt330%",
            "i5_c50_mb25_cntxt550%",
            "i5_c50_mb50_cntxt1100%",
        ],
    },
}

## Data Loading

In [3]:
def discover_data_params(data_path, max_samples=None):
    """Auto-discover ticker, date, and sample IDs from files in data_cond folder.
    Returns list of (ticker, date, sample_id) tuples to handle multiple dates."""
    cond_dir = data_path / "data_cond"
    pattern = re.compile(r"^(.+?)_(\d{4}-\d{2}-\d{2})_orderbook_real_id_(\d+)\.csv$")
    samples = []
    for f in cond_dir.glob("*_orderbook_real_id_*.csv"):
        match = pattern.match(f.name)
        if match:
            samples.append((match.group(1), match.group(2), int(match.group(3))))
    if not samples:
        raise ValueError(f"No orderbook files found in {cond_dir}")
    samples.sort()
    if max_samples is not None and len(samples) > max_samples:
        rng = np.random.RandomState(42)
        idx = rng.choice(len(samples), size=max_samples, replace=False)
        samples = [samples[i] for i in sorted(idx)]
    return samples


def is_midprice_outlier(book_array, max_midprice):
    """Check if any midprice in the sample exceeds the threshold."""
    midprice = (book_array[:, 0] + book_array[:, 2]) / 2
    return np.any(midprice > max_midprice) or np.any(midprice <= 0)


def load_folder_data(data_path, max_samples=None, max_midprice=None):
    """Load all data from a single folder, handling multiple dates."""
    samples = discover_data_params(data_path, max_samples=max_samples)
    gen_books, gen_msgs, cond_lens = {}, {}, {}
    n_outliers = 0
    for ticker, date, sid in samples:
        cond_book_path = data_path / f"data_cond/{ticker}_{date}_orderbook_real_id_{sid}.csv"
        gen_book_path = data_path / f"data_gen/{ticker}_{date}_orderbook_real_id_{sid}_gen_id_0.csv"
        gen_msg_path = data_path / f"data_gen/{ticker}_{date}_message_real_id_{sid}_gen_id_0.csv"
        cond_msg_path = data_path / f"data_cond/{ticker}_{date}_message_real_id_{sid}.csv"
        if not gen_book_path.exists():
            continue
        cond_book = np.loadtxt(cond_book_path, delimiter=',')
        gen_book = np.loadtxt(gen_book_path, delimiter=',')
        full_book = np.vstack([cond_book, gen_book])
        if max_midprice is not None and is_midprice_outlier(full_book, max_midprice):
            n_outliers += 1
            continue
        gen_msg = np.loadtxt(gen_msg_path, delimiter=',')
        cond_msg = np.loadtxt(cond_msg_path, delimiter=',')
        key = (date, sid)
        cond_lens[key] = cond_book.shape[0]
        gen_books[key] = full_book
        gen_msgs[key] = np.vstack([cond_msg, gen_msg])
    if not gen_books:
        raise ValueError(f"No complete sample pairs found in {data_path}")
    if n_outliers > 0:
        print(f"  filtered {n_outliers} outlier samples (midprice > {max_midprice})")
    return gen_books, gen_msgs, cond_lens


def load_buy_sell(context_length):
    """Load both buy and sell data for a given context length."""
    cfg = CONFIGS[context_length]
    all_data = {}
    for folder in cfg['folders']:
        buy_path = cfg['buy_path'] / folder
        sell_path = cfg['sell_path'] / folder
        if not buy_path.exists() or not sell_path.exists():
            print(f"SKIP: {folder} (missing buy or sell)")
            continue
        try:
            buy_books, buy_msgs, buy_cond = load_folder_data(
                buy_path, max_samples=MAX_SAMPLES, max_midprice=MIDPRICE_MAX)
            sell_books, sell_msgs, sell_cond = load_folder_data(
                sell_path, max_samples=MAX_SAMPLES, max_midprice=MIDPRICE_MAX)
            all_data[folder] = {
                'buy': {'books': buy_books, 'msgs': buy_msgs, 'cond_lens': buy_cond},
                'sell': {'books': sell_books, 'msgs': sell_msgs, 'cond_lens': sell_cond},
            }
            print(f"OK: {folder} (buy={len(buy_books)}, sell={len(sell_books)} samples)")
        except Exception as e:
            print(f"ERROR: {folder} - {e}")
    return all_data

In [4]:
data_500 = load_buy_sell(500)
data_250 = load_buy_sell(250)

KeyboardInterrupt: 

## Helpers

In [ ]:
def compute_midprice(book_array):
    """Compute midprice: (best_ask + best_bid) / 2."""
    return (book_array[:, 0] + book_array[:, 2]) / 2


def parse_folder_params(folder_name):
    """Parse iterations, coolings, metablock from folder name."""
    match = re.match(r'i(\d+)_c(\d+)_mb(\d+)_cntxt(.+)', folder_name)
    if match:
        return int(match.group(1)), int(match.group(2)), int(match.group(3))
    return None, None, None


def compute_midprice_returns(books, min_len):
    """Compute per-sample midprice returns (relative to first price)."""
    returns = []
    for sid, book_array in books.items():
        midprice = compute_midprice(book_array[:min_len])
        returns.append(midprice - midprice[0])
    return np.stack(returns, axis=0)


def compute_combined_impact(buy_data, sell_data, context_length, folder_name):
    """
    Combined impact = (buy_return - sell_return) / 2
    
    Buy pushes price UP (+), sell pushes price DOWN (-),
    so negating sell and averaging gives symmetric absolute impact.
    """
    iterations, coolings, metablock = parse_folder_params(folder_name)
    
    buy_books = buy_data['books']
    sell_books = sell_data['books']
    buy_msgs = buy_data['msgs']
    
    min_len = min(
        min(b.shape[0] for b in buy_books.values()),
        min(b.shape[0] for b in sell_books.values())
    )
    
    buy_returns = compute_midprice_returns(buy_books, min_len)
    sell_returns = compute_midprice_returns(sell_books, min_len)
    
    # Combined: (buy - sell) / 2
    combined = (buy_returns.mean(axis=0) - sell_returns.mean(axis=0)) / 2
    
    # Per-sample combined for std estimation
    # Match samples pairwise (same index), or use independent means
    buy_mean = buy_returns.mean(axis=0)
    sell_mean = sell_returns.mean(axis=0)
    buy_std = buy_returns.std(axis=0)
    sell_std = sell_returns.std(axis=0)
    n_buy, n_sell = buy_returns.shape[0], sell_returns.shape[0]
    # Propagated std: std((A-B)/2) = sqrt(std_A^2/n_A + std_B^2/n_B) / 2 * sqrt(n)
    # For the mean curves: SE = sqrt(se_buy^2 + se_sell^2) / 2
    combined_std = np.sqrt(buy_std**2 + sell_std**2) / 2
    
    junction = list(buy_data['cond_lens'].values())[0]
    cooling_start = context_length + iterations * metablock if iterations else None
    
    return {
        'steps': np.arange(min_len),
        'mean': combined,
        'std': combined_std,
        'buy_mean': buy_mean,
        'sell_mean': sell_mean,
        'junction': junction,
        'cooling_start': cooling_start,
        'iterations': iterations,
        'coolings': coolings,
        'metablock': metablock,
        'n_buy': n_buy,
        'n_sell': n_sell,
        'min_len': min_len,
    }

## 1. Combined Market Impact: All Configurations

In [ ]:
def compute_all_stats(all_data, context_length):
    """Compute combined stats for all folders."""
    stats = {}
    for folder, data in all_data.items():
        stats[folder] = compute_combined_impact(
            data['buy'], data['sell'], context_length, folder
        )
    return stats


def plot_combined_impact(stats_dict, context_length):
    """Plot combined (buy-sell)/2 midprice impact for all folders."""
    if not stats_dict:
        print(f"No data to plot for context={context_length}")
        return
    
    fig = go.Figure()
    colors = px.colors.qualitative.Set1 + px.colors.qualitative.Set2
    
    for idx, (folder, s) in enumerate(stats_dict.items()):
        color = colors[idx % len(colors)]
        steps = s['steps']
        
        # Shaded std region
        fig.add_trace(go.Scatter(
            x=np.concatenate([steps, steps[::-1]]),
            y=np.concatenate([s['mean'] + s['std'], (s['mean'] - s['std'])[::-1]]),
            fill='toself',
            fillcolor=color.replace('rgb', 'rgba').replace(')', ', 0.12)'),
            line=dict(color='rgba(255,255,255,0)'),
            showlegend=False, hoverinfo='skip'
        ))
        
        # Mean line
        fig.add_trace(go.Scatter(
            x=steps, y=s['mean'], mode='lines',
            name=f"{folder} (n={s['n_buy']}+{s['n_sell']})",
            line=dict(color=color, width=2),
        ))
        
        # Cooling start marker
        ct = s['cooling_start']
        if ct and ct < len(s['mean']):
            fig.add_trace(go.Scatter(
                x=[ct], y=[s['mean'][ct]], mode='markers',
                marker=dict(size=10, symbol='circle', color=color,
                            line=dict(color='black', width=1.5)),
                showlegend=False,
                hovertemplate=f"{folder}<br>Cooling start: {ct}<br>impact=%{{y:.2f}}<extra></extra>"
            ))
    
    # Junction line
    junction = list(stats_dict.values())[0]['junction']
    fig.add_vline(x=junction, line_color="gray", line_width=2, line_dash="dash",
                  annotation_text="Junction", annotation_position="top")
    fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=1)
    
    fig.update_layout(
        title=f'Combined Market Impact (buy - sell) / 2 | context={context_length}',
        xaxis_title='Steps', yaxis_title='Impact (price units)',
        width=1200, height=600, template='plotly_white',
        legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.8)')
    )
    fig.show()


stats_500 = compute_all_stats(data_500, 500)
stats_250 = compute_all_stats(data_250, 250)

plot_combined_impact(stats_500, 500)
plot_combined_impact(stats_250, 250)

## 2. Grouped by Iterations (i3 vs i5)

In [ ]:
def plot_grouped_by_iterations(stats_dict, context_length):
    """Two subplots: i3 and i5, colored by metablock."""
    groups = {'i3': {}, 'i5': {}}
    for folder, s in stats_dict.items():
        if folder.startswith('i3_'):
            groups['i3'][folder] = s
        elif folder.startswith('i5_'):
            groups['i5'][folder] = s
    
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=[f'i=3 (3 insertions)', f'i=5 (5 insertions)'],
        horizontal_spacing=0.08
    )
    
    colors = px.colors.qualitative.Set1
    
    for col, (group_name, group_stats) in enumerate(groups.items(), 1):
        for idx, (folder, s) in enumerate(sorted(group_stats.items())):
            mb_match = re.search(r'mb(\d+)', folder)
            mb_val = mb_match.group(1) if mb_match else folder
            legend_group = f"mb{mb_val}"
            color = colors[idx % len(colors)]
            
            # Std band
            fig.add_trace(go.Scatter(
                x=np.concatenate([s['steps'], s['steps'][::-1]]),
                y=np.concatenate([s['mean'] + s['std'], (s['mean'] - s['std'])[::-1]]),
                fill='toself',
                fillcolor=color.replace('rgb', 'rgba').replace(')', ', 0.12)'),
                line=dict(color='rgba(255,255,255,0)'),
                legendgroup=legend_group, showlegend=False, hoverinfo='skip'
            ), row=1, col=col)
            
            # Mean
            fig.add_trace(go.Scatter(
                x=s['steps'], y=s['mean'], mode='lines',
                name=legend_group, line=dict(color=color, width=2),
                legendgroup=legend_group, showlegend=(col == 1)
            ), row=1, col=col)
            
            # Cooling marker
            ct = s['cooling_start']
            if ct and ct < len(s['mean']):
                fig.add_trace(go.Scatter(
                    x=[ct], y=[s['mean'][ct]], mode='markers',
                    marker=dict(size=10, symbol='circle', color=color,
                                line=dict(color='black', width=1.5)),
                    legendgroup=legend_group, showlegend=False,
                ), row=1, col=col)
        
        # Junction
        if group_stats:
            junction = list(group_stats.values())[0]['junction']
            fig.add_vline(x=junction, line_color="gray", line_dash="dash", row=1, col=col)
        fig.add_hline(y=0, line_color="gray", line_dash="dot", row=1, col=col)
    
    fig.update_layout(
        title=f'Combined Impact by Iterations | context={context_length}',
        width=1200, height=500, template='plotly_white',
    )
    fig.show()


plot_grouped_by_iterations(stats_500, 500)
plot_grouped_by_iterations(stats_250, 250)

## 3. Market Impact Decay: Cooling Phase Analysis

After the last aggressive order (cooling_start), the price should partially revert.  
Theory predicts power-law decay: $\Delta p(t) \sim t^{-\gamma}$, typically $\gamma \approx 0.5$.

Here we normalize the cooling phase: set t=0 at cooling_start, normalize impact to 1.0 at peak.

In [ ]:
def extract_decay_curve(stats):
    """
    Extract normalized decay curve from cooling phase.
    Returns (t, normalized_impact) where t starts at 1 and impact starts at 1.0.
    """
    ct = stats['cooling_start']
    if ct is None or ct >= len(stats['mean']):
        return None, None
    
    cooling_impact = stats['mean'][ct:]
    peak = cooling_impact[0]
    
    if peak <= 0:
        return None, None
    
    normalized = cooling_impact / peak
    t = np.arange(1, len(normalized) + 1)
    
    return t, normalized


def fit_power_law(t, y, fit_range=None):
    """Fit y = a * t^(-gamma) in log-log space. Returns (gamma, a, r_squared)."""
    if fit_range is not None:
        mask = (t >= fit_range[0]) & (t <= fit_range[1])
        t, y = t[mask], y[mask]
    
    valid = (t > 0) & (y > 0) & np.isfinite(y)
    if valid.sum() < 3:
        return None, None, None
    
    log_t = np.log(t[valid])
    log_y = np.log(y[valid])
    
    # Linear fit: log(y) = log(a) - gamma * log(t)
    coeffs = np.polyfit(log_t, log_y, 1)
    gamma = -coeffs[0]
    a = np.exp(coeffs[1])
    
    # R-squared
    predicted = coeffs[0] * log_t + coeffs[1]
    ss_res = np.sum((log_y - predicted) ** 2)
    ss_tot = np.sum((log_y - log_y.mean()) ** 2)
    r_sq = 1 - ss_res / ss_tot if ss_tot > 0 else 0
    
    return gamma, a, r_sq

In [ ]:
def plot_decay_curves(stats_dict, context_length):
    """Plot normalized decay curves for all folders, with power-law fits."""
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=['Linear scale', 'Log-log scale'],
        horizontal_spacing=0.1
    )
    
    colors = px.colors.qualitative.Set1 + px.colors.qualitative.Set2
    fit_results = []
    
    for idx, (folder, s) in enumerate(sorted(stats_dict.items())):
        t, norm_impact = extract_decay_curve(s)
        if t is None:
            continue
        
        color = colors[idx % len(colors)]
        iterations, coolings, mb = s['iterations'], s['coolings'], s['metablock']
        label = f"i{iterations}_mb{mb}"
        
        # Fit power law (skip first few steps for stability)
        gamma, a, r_sq = fit_power_law(t, norm_impact, fit_range=(3, len(t)))
        if gamma is not None:
            fit_results.append({
                'folder': folder, 'gamma': gamma, 'r_squared': r_sq,
                'iterations': iterations, 'metablock': mb,
                'cooling_steps': coolings * mb,
            })
            label += f" (gamma={gamma:.2f})"
        
        # Linear scale
        fig.add_trace(go.Scatter(
            x=t, y=norm_impact, mode='lines',
            name=label, line=dict(color=color, width=2),
            legendgroup=folder, showlegend=True,
        ), row=1, col=1)
        
        # Log-log scale
        fig.add_trace(go.Scatter(
            x=t, y=norm_impact, mode='lines',
            line=dict(color=color, width=2),
            legendgroup=folder, showlegend=False,
        ), row=1, col=2)
        
        # Power-law fit line on log-log
        if gamma is not None:
            fit_t = np.linspace(3, len(t), 100)
            fit_y = a * fit_t ** (-gamma)
            fig.add_trace(go.Scatter(
                x=fit_t, y=fit_y, mode='lines',
                line=dict(color=color, width=1, dash='dot'),
                legendgroup=folder, showlegend=False,
            ), row=1, col=2)
    
    # Reference power-law curves on log-log subplot
    ref_t = np.linspace(3, 500, 100)
    fig.add_trace(go.Scatter(
        x=ref_t, y=ref_t**(-0.3), mode='lines',
        name='t^(-0.3) reference', line=dict(color='gray', width=2, dash='dot'),
    ), row=1, col=2)
    fig.add_trace(go.Scatter(
        x=ref_t, y=ref_t**(-0.4), mode='lines',
        name='t^(-0.4) reference', line=dict(color='black', width=2, dash='dash'),
    ), row=1, col=2)
    fig.add_trace(go.Scatter(
        x=ref_t, y=ref_t**(-0.5), mode='lines',
        name='t^(-0.5) reference', line=dict(color='black', width=2, dash='dash'),
    ), row=1, col=2)
    
    fig.update_xaxes(type='log', row=1, col=2)
    fig.update_yaxes(type='log', row=1, col=2)
    fig.update_xaxes(title_text='Steps since cooling start', row=1, col=1)
    fig.update_xaxes(title_text='Steps since cooling start (log)', row=1, col=2)
    fig.update_yaxes(title_text='Normalized impact', row=1, col=1)
    fig.update_yaxes(title_text='Normalized impact (log)', row=1, col=2)
    
    fig.update_layout(
        title=f'Market Impact Decay During Cooling Phase | context={context_length}',
        width=1300, height=550, template='plotly_white',
    )
    fig.show()
    
    return pd.DataFrame(fit_results)


decay_500 = plot_decay_curves(stats_500, 500)
decay_250 = plot_decay_curves(stats_250, 250)

## 4. Decay Exponent Summary

In [ ]:
def print_gamma_table(decay_df, context_length):
    """Print gamma exponents as pivot table."""
    if decay_df.empty:
        print(f"No decay data for context={context_length}")
        return
    
    print(f"=== Decay Exponent gamma | context={context_length} ===")
    print(f"(Theory: gamma ~ 0.5)\n")
    
    pivot = decay_df.pivot(index='iterations', columns='metablock', values='gamma')
    pivot.index = [f'i{i}' for i in pivot.index]
    pivot.columns = [f'mb{c}' for c in pivot.columns]
    print(pivot.round(3).to_string())
    
    print(f"\n=== R-squared ===")
    pivot_r = decay_df.pivot(index='iterations', columns='metablock', values='r_squared')
    pivot_r.index = [f'i{i}' for i in pivot_r.index]
    pivot_r.columns = [f'mb{c}' for c in pivot_r.columns]
    print(pivot_r.round(3).to_string())


print_gamma_table(decay_500, 500)
print()
print_gamma_table(decay_250, 250)

## 5. Decay vs Generation Horizon

Does the model maintain realistic decay for longer generation horizons?  
x-axis: generation horizon as % of context (cntxt%), y-axis: decay exponent gamma.

In [ ]:
def plot_gamma_vs_horizon(decay_500, decay_250):
    """Scatter: gamma vs cntxt% for both context lengths."""
    fig = go.Figure()
    
    for decay_df, ctx_len, symbol in [(decay_500, 500, 'circle'), (decay_250, 250, 'diamond')]:
        if decay_df.empty:
            continue
        for _, row in decay_df.iterrows():
            i, c, mb = parse_folder_params(row['folder'])
            cntxt_pct = (i + c) * mb / ctx_len * 100
            row['cntxt_pct'] = cntxt_pct
        
        decay_df['cntxt_pct'] = decay_df['folder'].apply(
            lambda f: (lambda p: (p[0]+p[1])*p[2]/ctx_len*100)(parse_folder_params(f))
        )
        
        for it_val in decay_df['iterations'].unique():
            sub = decay_df[decay_df['iterations'] == it_val]
            fig.add_trace(go.Scatter(
                x=sub['cntxt_pct'], y=sub['gamma'],
                mode='markers+text',
                marker=dict(size=12, symbol=symbol),
                text=[f"mb{r['metablock']}" for _, r in sub.iterrows()],
                textposition='top center',
                name=f'ctx={ctx_len}, i={it_val}',
            ))
    
    # Reference line gamma=0.5
    fig.add_hline(y=0.5, line_dash="dash", line_color="red", line_width=2,
                  annotation_text="gamma=0.5 (theory)", annotation_position="bottom right")
    
    fig.update_layout(
        title='Decay Exponent vs Generation Horizon',
        xaxis_title='Generation horizon (% of context)',
        yaxis_title='Decay exponent gamma',
        width=900, height=500, template='plotly_white',
    )
    fig.show()


plot_gamma_vs_horizon(decay_500, decay_250)

## 6. Peak Impact vs Generation Horizon

How does the peak impact (at cooling_start) scale with the number of aggressive insertions and metablock size?

In [ ]:
def plot_peak_impact(stats_500, stats_250):
    """Plot peak impact at cooling start vs cntxt%."""
    fig = go.Figure()
    
    for stats_dict, ctx_len, symbol in [(stats_500, 500, 'circle'), (stats_250, 250, 'diamond')]:
        rows = []
        for folder, s in stats_dict.items():
            ct = s['cooling_start']
            if ct is None or ct >= len(s['mean']):
                continue
            i, c, mb = s['iterations'], s['coolings'], s['metablock']
            cntxt_pct = (i + c) * mb / ctx_len * 100
            rows.append({
                'folder': folder, 'peak': s['mean'][ct],
                'iterations': i, 'metablock': mb, 'cntxt_pct': cntxt_pct,
            })
        df = pd.DataFrame(rows)
        
        for it_val in df['iterations'].unique():
            sub = df[df['iterations'] == it_val].sort_values('cntxt_pct')
            fig.add_trace(go.Scatter(
                x=sub['cntxt_pct'], y=sub['peak'],
                mode='markers+lines+text',
                marker=dict(size=10, symbol=symbol),
                text=[f"mb{r['metablock']}" for _, r in sub.iterrows()],
                textposition='top center',
                name=f'ctx={ctx_len}, i={it_val}',
            ))
    
    fig.update_layout(
        title='Peak Impact at Cooling Start vs Generation Horizon',
        xaxis_title='Generation horizon (% of context)',
        yaxis_title='Peak combined impact (price units)',
        width=900, height=500, template='plotly_white',
    )
    fig.show()


plot_peak_impact(stats_500, stats_250)

## 7. Reversion Ratio: How Much Impact Decays?

reversion_ratio = (peak_impact - final_impact) / peak_impact  
1.0 = full reversion (price returns to pre-impact), 0.0 = permanent impact.

In [ ]:
def compute_reversion_table(stats_dict, context_length):
    """Compute reversion ratio for all folders."""
    rows = []
    for folder, s in stats_dict.items():
        ct = s['cooling_start']
        if ct is None or ct >= len(s['mean']):
            continue
        
        peak = s['mean'][ct]
        final = s['mean'][-1]
        
        if peak > 0:
            reversion = (peak - final) / peak
        else:
            reversion = np.nan
        
        rows.append({
            'folder': folder,
            'iterations': s['iterations'],
            'metablock': s['metablock'],
            'peak_impact': peak,
            'final_impact': final,
            'reversion_ratio': reversion,
            'cooling_steps': s['coolings'] * s['metablock'],
        })
    
    df = pd.DataFrame(rows)
    
    print(f"=== Reversion Ratio | context={context_length} ===")
    print(f"(1.0 = full reversion, 0.0 = permanent impact)\n")
    
    if not df.empty:
        pivot = df.pivot(index='iterations', columns='metablock', values='reversion_ratio')
        pivot.index = [f'i{i}' for i in pivot.index]
        pivot.columns = [f'mb{c}' for c in pivot.columns]
        print(pivot.round(3).to_string())
        
        print(f"\n=== Peak Impact (price units) ===")
        pivot_peak = df.pivot(index='iterations', columns='metablock', values='peak_impact')
        pivot_peak.index = [f'i{i}' for i in pivot_peak.index]
        pivot_peak.columns = [f'mb{c}' for c in pivot_peak.columns]
        print(pivot_peak.round(1).to_string())
    
    return df


rev_500 = compute_reversion_table(stats_500, 500)
print()
rev_250 = compute_reversion_table(stats_250, 250)

## 8. Context 500 vs 250: Same Config Comparison

For matching (i, mb) pairs: does longer context produce more realistic decay?

In [ ]:
def plot_context_comparison(stats_500, stats_250):
    """Compare decay for same (i, mb) across context lengths."""
    # Find matching (i, mb) pairs
    pairs_500 = {(s['iterations'], s['metablock']): (folder, s) for folder, s in stats_500.items()}
    pairs_250 = {(s['iterations'], s['metablock']): (folder, s) for folder, s in stats_250.items()}
    
    common_keys = sorted(set(pairs_500.keys()) & set(pairs_250.keys()))
    if not common_keys:
        print("No matching (i, mb) pairs between context 500 and 250")
        return
    
    n_pairs = len(common_keys)
    cols = min(4, n_pairs)
    rows = (n_pairs + cols - 1) // cols
    
    fig = make_subplots(
        rows=rows, cols=cols,
        subplot_titles=[f'i={k[0]}, mb={k[1]}' for k in common_keys],
    )
    
    for idx, key in enumerate(common_keys):
        r = idx // cols + 1
        c = idx % cols + 1
        
        for stats_dict, ctx_len, color in [(stats_500, 500, 'blue'), (stats_250, 250, 'red')]:
            pairs = {(s['iterations'], s['metablock']): (folder, s) for folder, s in stats_dict.items()}
            if key not in pairs:
                continue
            folder, s = pairs[key]
            t, norm = extract_decay_curve(s)
            if t is None:
                continue
            
            fig.add_trace(go.Scatter(
                x=t, y=norm, mode='lines',
                name=f'ctx={ctx_len}', line=dict(color=color, width=2),
                legendgroup=f'ctx{ctx_len}', showlegend=(idx == 0),
            ), row=r, col=c)
        
        fig.update_xaxes(type='log', row=r, col=c)
        fig.update_yaxes(type='log', row=r, col=c)
    
    fig.update_layout(
        title='Normalized Decay: Context 500 (blue) vs 250 (red)',
        width=1200, height=300 * rows, template='plotly_white',
    )
    fig.show()


plot_context_comparison(stats_500, stats_250)

## 9. Recommendations

**Hypothesis:** The model was trained to generate 500 messages conditioned on 500. Market impact decay should be reproduced when total generated messages stay within the training budget (~500), and degrade beyond it.

In [ ]:
# ── Build combined results DataFrame ──────────────────────────────────────────

TRAINING_BUDGET = 500  # model was trained to generate 500 messages on 500 context

# Context descriptions for printing
CTX_INFO = {
    500: "cooling ratio ~3-4x  (shorter cooling relative to context — gamma less reliable)",
    250: "cooling ratio ~10x   (longer cooling relative to context — gamma more reliable)",
}

def build_combined_results(stats_500, stats_250, decay_500, decay_250):
    """Build a single DataFrame with all config results."""
    rows = []
    for stats_dict, decay_df, ctx in [(stats_500, decay_500, 500), (stats_250, decay_250, 250)]:
        for folder, s in stats_dict.items():
            ct = s['cooling_start']
            if ct is None or ct >= len(s['mean']):
                continue

            peak = s['mean'][ct]
            final = s['mean'][-1]
            reversion = (peak - final) / peak if peak > 0 else np.nan

            gamma_row = decay_df[decay_df['folder'] == folder] if not decay_df.empty else pd.DataFrame()
            gamma = gamma_row['gamma'].values[0] if len(gamma_row) > 0 else np.nan
            r_sq = gamma_row['r_squared'].values[0] if len(gamma_row) > 0 else np.nan

            i, c, mb = s['iterations'], s['coolings'], s['metablock']
            gen_msgs_aggressive = i * mb
            gen_msgs_cooling = c * mb
            gen_msgs_total = gen_msgs_aggressive + gen_msgs_cooling

            rows.append({
                'context': ctx,
                'folder': folder,
                'iterations': i,
                'metablock': mb,
                'cooling_steps': gen_msgs_cooling,
                'gen_aggressive': gen_msgs_aggressive,
                'gen_total': gen_msgs_total,
                'within_budget': gen_msgs_total <= TRAINING_BUDGET,
                'gamma': gamma,
                'r_squared': r_sq,
                'peak_impact': peak,
                'reversion_ratio': reversion,
                'n_buy': s['n_buy'],
                'n_sell': s['n_sell'],
            })

    return pd.DataFrame(rows)


results = build_combined_results(stats_500, stats_250, decay_500, decay_250)

# ── Verdict per experiment (A) ────────────────────────────────────────────────
# Focus on decay speed (gamma) — NOT reversion_ratio (depends on unreliable final value)

def classify_verdict(row):
    """Per-experiment verdict based on gamma, R², and peak only."""
    g = row['gamma']
    r2 = row['r_squared']
    peak = row['peak_impact']
    if pd.isna(g) or pd.isna(r2) or pd.isna(peak) or peak <= 0:
        return 'does not reproduce'
    if 0.25 <= g <= 0.55 and r2 > 0.5 and peak > 0:
        return 'reproduces'
    if 0.15 <= g <= 0.65 and r2 > 0.3 and peak > 0:
        return 'marginal'
    return 'does not reproduce'


results['verdict'] = results.apply(classify_verdict, axis=1)

# Score 0-3: gamma in range, R²>0.5, peak>0  (no reversion_ratio)
def score_config(row):
    score = 0
    if pd.notna(row['gamma']) and 0.25 <= row['gamma'] <= 0.55:
        score += 1
    if pd.notna(row['r_squared']) and row['r_squared'] > 0.5:
        score += 1
    if pd.notna(row['peak_impact']) and row['peak_impact'] > 0:
        score += 1
    return score

results['score'] = results.apply(score_config, axis=1)

# ══════════════════════════════════════════════════════════════════════════════
# (B) PER-EXPERIMENT TABLE — always split by context
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 110)
print("PER-EXPERIMENT VERDICTS  (scored 0-3: gamma in [0.25,0.55], R²>0.5, peak>0)")
print("  reproduces         = gamma in [0.25, 0.55] AND R² > 0.5 AND peak > 0")
print("  marginal           = gamma in [0.15, 0.65] AND R² > 0.3 AND peak > 0")
print("  does not reproduce = everything else")
print("=" * 110)

for ctx in [500, 250]:
    sub = results[results['context'] == ctx].sort_values('gen_total')
    print(f"\n{'=' * 110}")
    print(f"  CONTEXT {ctx}  ({CTX_INFO[ctx]})")
    print(f"{'=' * 110}")
    fmt = "  {folder:32s} {gen_total:>5d}  {gamma:>7s}  {r2:>7s}  {peak:>7s}  {verdict:s}"
    print(fmt.format(folder="folder", gen_total=0, gamma="gamma", r2="R²",
                     peak="peak", verdict="verdict").replace("    0", "gen_t"))
    print("  " + "-" * 100)
    for _, r in sub.iterrows():
        g_str = f"{r['gamma']:.3f}" if pd.notna(r['gamma']) else "  N/A"
        r2_str = f"{r['r_squared']:.3f}" if pd.notna(r['r_squared']) else "  N/A"
        p_str = f"{r['peak_impact']:.1f}" if pd.notna(r['peak_impact']) else "  N/A"
        print(fmt.format(folder=r['folder'], gen_total=int(r['gen_total']),
                         gamma=g_str, r2=r2_str, peak=p_str, verdict=r['verdict']))

# ══════════════════════════════════════════════════════════════════════════════
# (C) PER-CONTEXT SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 110)
print("PER-CONTEXT SUMMARY")
print("=" * 110)

for ctx in [500, 250]:
    sub = results[results['context'] == ctx]
    n_total = len(sub)
    n_repr = (sub['verdict'] == 'reproduces').sum()
    n_marg = (sub['verdict'] == 'marginal').sum()
    n_fail = (sub['verdict'] == 'does not reproduce').sum()

    print(f"\n  CONTEXT {ctx}  ({CTX_INFO[ctx]})")
    print(f"    {n_repr} reproduces / {n_marg} marginal / {n_fail} does not reproduce  "
          f"(out of {n_total})")

    repr_cfgs = sub[sub['verdict'] == 'reproduces']
    if not repr_cfgs.empty:
        mb_range = f"{repr_cfgs['metablock'].min()}-{repr_cfgs['metablock'].max()}"
        gen_range = f"{repr_cfgs['gen_total'].min()}-{repr_cfgs['gen_total'].max()}"
        print(f"    Reproducing configs: {list(repr_cfgs['folder'])}")
        print(f"      metablock range: {mb_range},  gen_total range: {gen_range}")
    else:
        print(f"    No configs reproduce impact decay for this context")

    # Where does quality degrade?
    sorted_sub = sub.sort_values('gen_total')
    fail_cfgs = sorted_sub[sorted_sub['verdict'] == 'does not reproduce']
    if not fail_cfgs.empty:
        first_fail_gen = fail_cfgs['gen_total'].min()
        first_fail_mb = fail_cfgs[fail_cfgs['gen_total'] == first_fail_gen]['metablock'].values[0]
        print(f"    Quality degrades at gen_total >= {first_fail_gen} (mb={first_fail_mb})")

# ══════════════════════════════════════════════════════════════════════════════
# (D) BREAKDOWNS — always split by context
# ══════════════════════════════════════════════════════════════════════════════

# ── BY ITERATIONS (per context) ───────────────────────────────────────────────

print("\n" + "-" * 110)
print("BY ITERATIONS (per context)")
for ctx in [500, 250]:
    sub = results[results['context'] == ctx]
    print(f"\n  context={ctx}:")
    for it in sorted(sub['iterations'].unique()):
        ss = sub[sub['iterations'] == it]
        avg_g = ss['gamma'].mean()
        avg_peak = ss['peak_impact'].mean()
        n_repr = (ss['verdict'] == 'reproduces').sum()
        print(f"    i={it}:  avg gamma={avg_g:.3f}  avg peak={avg_peak:.1f}  "
              f"reproduces: {n_repr}/{len(ss)}  "
              f"aggressive msgs: {ss['gen_aggressive'].min()}-{ss['gen_aggressive'].max()}")

# ── BY METABLOCK (per context) ────────────────────────────────────────────────

print("\n" + "-" * 110)
print("BY METABLOCK (per context)")
for ctx in [500, 250]:
    sub = results[results['context'] == ctx]
    print(f"\n  context={ctx}:")
    for mb in sorted(sub['metablock'].unique()):
        ss = sub[sub['metablock'] == mb]
        avg_g = ss['gamma'].mean()
        avg_r2 = ss['r_squared'].mean()
        n_repr = (ss['verdict'] == 'reproduces').sum()
        print(f"    mb={mb:2d}:  avg gamma={avg_g:.3f}  avg R²={avg_r2:.3f}  "
              f"reproduces: {n_repr}/{len(ss)}")

# ── GENERATION HORIZON (per context) ──────────────────────────────────────────

print("\n" + "-" * 110)
print("GENERATION HORIZON (per context)")
for ctx in [500, 250]:
    sub = results[results['context'] == ctx]
    good = sub[sub['verdict'] == 'reproduces']
    print(f"\n  context={ctx}:")
    if not good.empty:
        print(f"    Configs reproducing: {len(good)}/{len(sub)}")
        print(f"      gen_total range: {good['gen_total'].min()}-{good['gen_total'].max()}")
        print(f"      max gen_total with reproduces: {good['gen_total'].max()}")
    else:
        print(f"    No configs reproduce impact decay")

# ── COOLING PHASE (per context) ───────────────────────────────────────────────

print("\n" + "-" * 110)
print("COOLING PHASE (per context)")
for ctx in [500, 250]:
    sub = results[results['context'] == ctx]
    good_cool = sub[sub['r_squared'] > 0.5]
    print(f"\n  context={ctx}:")
    if not good_cool.empty:
        print(f"    Min cooling msgs with R²>0.5: {good_cool['cooling_steps'].min()}")
        print(f"    Configs with R²>0.5: {len(good_cool)}/{len(sub)}")
    else:
        print(f"    No configs achieved R²>0.5")

# ── ANALYSIS BY GENERATED MESSAGES (per context) ─────────────────────────────

print("\n" + "=" * 110)
print("ANALYSIS BY GENERATED MESSAGES (per context, training budget = 500 messages)")
print("=" * 110)

for ctx in [500, 250]:
    sub = results[results['context'] == ctx]
    print(f"\n  context={ctx}  ({CTX_INFO[ctx]})")
    for label, mask in [("WITHIN budget (<=500 msgs)", sub['within_budget']),
                        ("OVER budget (>500 msgs)", ~sub['within_budget'])]:
        ss = sub[mask]
        if ss.empty:
            print(f"    {label}: no configs")
            continue
        n = len(ss)
        avg_g = ss['gamma'].mean()
        avg_r2 = ss['r_squared'].mean()
        n_repr = (ss['verdict'] == 'reproduces').sum()
        n_marg = (ss['verdict'] == 'marginal').sum()
        print(f"    {label}  ({n} configs)")
        print(f"      avg gamma={avg_g:.3f}  avg R²={avg_r2:.3f}  "
              f"reproduces: {n_repr}  marginal: {n_marg}")

# ══════════════════════════════════════════════════════════════════════════════
# (E) BEST OPERATING POINT — per context
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 110)
print("BEST OPERATING POINT (per context)")
print("=" * 110)

for ctx in [500, 250]:
    sub = results[results['context'] == ctx]
    repr_cfgs = sub[sub['verdict'] == 'reproduces'].sort_values('score', ascending=False)

    print(f"\n  CONTEXT {ctx}  ({CTX_INFO[ctx]})")

    if repr_cfgs.empty:
        # Fall back to marginal
        marg = sub[sub['verdict'] == 'marginal'].sort_values('score', ascending=False)
        if marg.empty:
            print("    No configs reproduce or are marginal for impact decay")
        else:
            best = marg.iloc[0]
            print(f"    No reproducing configs — best marginal:")
            print(f"      {best['folder']}  gen_total={int(best['gen_total'])}  "
                  f"gamma={best['gamma']:.3f}  R²={best['r_squared']:.3f}")
    else:
        best = repr_cfgs.iloc[0]
        print(f"    Best: {best['folder']}")
        print(f"      Insertions:      i{int(best['iterations'])}  "
              f"({int(best['gen_aggressive'])} aggressive msgs)")
        print(f"      Metablock:       mb{int(best['metablock'])}")
        print(f"      Total generated: {int(best['gen_total'])} messages")
        print(f"      gamma={best['gamma']:.3f}  R²={best['r_squared']:.3f}  "
              f"peak={best['peak_impact']:.1f}")
        # List all reproducing
        if len(repr_cfgs) > 1:
            print(f"    All reproducing ({len(repr_cfgs)}):")
            for _, r in repr_cfgs.iterrows():
                print(f"      {r['folder']:32s}  gen={int(r['gen_total']):4d}  "
                      f"gamma={r['gamma']:.3f}  R²={r['r_squared']:.3f}")

# ══════════════════════════════════════════════════════════════════════════════
# POOR CONFIGS (per context)
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "-" * 110)
print("DOES NOT REPRODUCE — per context")
for ctx in [500, 250]:
    sub = results[(results['context'] == ctx) & (results['verdict'] == 'does not reproduce')]
    print(f"\n  context={ctx}  ({len(sub)} configs):")
    if sub.empty:
        print("    (none)")
    else:
        for _, r in sub.sort_values('gen_total').iterrows():
            reason_parts = []
            if pd.isna(r['gamma']):
                reason_parts.append("no fit")
            elif not (0.15 <= r['gamma'] <= 0.65):
                reason_parts.append(f"gamma={r['gamma']:.3f} out of range")
            if pd.isna(r['r_squared']):
                reason_parts.append("no R²")
            elif r['r_squared'] <= 0.3:
                reason_parts.append(f"R²={r['r_squared']:.3f} too low")
            if pd.isna(r['peak_impact']) or r['peak_impact'] <= 0:
                reason_parts.append(f"peak={r['peak_impact']:.1f} (no impact)")
            reason = "; ".join(reason_parts) if reason_parts else "borderline"
            print(f"    {r['folder']:32s}  gen={int(r['gen_total']):4d}  [{reason}]")

print("\n" + "=" * 110)

In [ ]:
# ── Recommendations Visualizations (2x2) ─────────────────────────────────────

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'gamma by Metablock (grouped by iterations)',
        'R² by Metablock (grouped by iterations)',
        'Reversion Ratio by Metablock',
        'gamma vs Total Generated Messages',
    ],
    vertical_spacing=0.14,
    horizontal_spacing=0.1,
)

colors_iter = {'i3': '#636EFA', 'i5': '#EF553B'}

# ── (1,1) gamma by metablock, grouped by iterations ──
for it in sorted(results['iterations'].unique()):
    sub = results[results['iterations'] == it].groupby('metablock')['gamma'].mean().reset_index()
    fig.add_trace(go.Bar(
        x=[f'mb{m}' for m in sub['metablock']],
        y=sub['gamma'],
        name=f'i{it}',
        marker_color=colors_iter[f'i{it}'],
        legendgroup=f'i{it}',
        showlegend=True,
    ), row=1, col=1)
fig.add_hline(y=0.3, line_dash='dot', line_color='gray', line_width=1.5, row=1, col=1,
              annotation_text='0.3', annotation_position='bottom left',
              annotation_font_size=10, annotation_font_color='gray')
fig.add_hline(y=0.4, line_dash='dash', line_color='black', line_width=1.5, row=1, col=1,
              annotation_text='0.4', annotation_position='top left',
              annotation_font_size=10)
fig.add_hline(y=0.5, line_dash='dot', line_color='gray', line_width=1.5, row=1, col=1,
              annotation_text='0.5', annotation_position='top left',
              annotation_font_size=10, annotation_font_color='gray')

# ── (1,2) R² by metablock, grouped by iterations ──
for it in sorted(results['iterations'].unique()):
    sub = results[results['iterations'] == it].groupby('metablock')['r_squared'].mean().reset_index()
    fig.add_trace(go.Bar(
        x=[f'mb{m}' for m in sub['metablock']],
        y=sub['r_squared'],
        name=f'i{it}',
        marker_color=colors_iter[f'i{it}'],
        legendgroup=f'i{it}',
        showlegend=False,
    ), row=1, col=2)
fig.add_hline(y=0.5, line_dash='dash', line_color='red', line_width=1.5, row=1, col=2,
              annotation_text='R²=0.5', annotation_position='bottom right',
              annotation_font_size=10, annotation_font_color='red')

# ── (2,1) Reversion ratio by metablock ──
for it in sorted(results['iterations'].unique()):
    sub = results[results['iterations'] == it].groupby('metablock')['reversion_ratio'].mean().reset_index()
    fig.add_trace(go.Bar(
        x=[f'mb{m}' for m in sub['metablock']],
        y=sub['reversion_ratio'],
        name=f'i{it}',
        marker_color=colors_iter[f'i{it}'],
        legendgroup=f'i{it}',
        showlegend=False,
    ), row=2, col=1)
fig.add_hrect(y0=0.3, y1=0.8, fillcolor='green', opacity=0.08, line_width=0, row=2, col=1)
fig.add_hline(y=0.3, line_dash='dot', line_color='green', line_width=1, row=2, col=1)
fig.add_hline(y=0.8, line_dash='dot', line_color='green', line_width=1, row=2, col=1)

# ── (2,2) gamma vs generated messages with budget line + good zone ──
fig.add_hrect(y0=0.3, y1=0.5, fillcolor='green', opacity=0.1, line_width=0, row=2, col=2)
fig.add_hline(y=0.3, line_dash='dot', line_color='gray', line_width=1.5, row=2, col=2,
              annotation_text='0.3', annotation_position='bottom right',
              annotation_font_size=10, annotation_font_color='gray')
fig.add_hline(y=0.4, line_dash='dash', line_color='black', line_width=1.5, row=2, col=2,
              annotation_text='0.4', annotation_position='top right',
              annotation_font_size=10)
fig.add_hline(y=0.5, line_dash='dot', line_color='gray', line_width=1.5, row=2, col=2,
              annotation_text='0.5', annotation_position='top right',
              annotation_font_size=10, annotation_font_color='gray')
# Training budget vertical line
fig.add_vline(x=TRAINING_BUDGET, line_dash='dash', line_color='red', line_width=2, row=2, col=2,
              annotation_text=f'training budget ({TRAINING_BUDGET} msgs)',
              annotation_position='top left', annotation_font_size=10,
              annotation_font_color='red')

for ctx in [500, 250]:
    sub = results[results['context'] == ctx].sort_values('gen_total')
    for it in sorted(sub['iterations'].unique()):
        ss = sub[sub['iterations'] == it]
        fig.add_trace(go.Scatter(
            x=ss['gen_total'], y=ss['gamma'],
            mode='markers+lines+text',
            marker=dict(size=8, symbol='circle' if ctx == 500 else 'diamond'),
            text=[f'mb{r["metablock"]}' for _, r in ss.iterrows()],
            textposition='top center', textfont_size=9,
            name=f'ctx={ctx}, i{it}',
            legendgroup=f'ctx{ctx}_i{it}',
            showlegend=True,
        ), row=2, col=2)

# ── Layout ──
fig.update_yaxes(title_text='gamma', row=1, col=1)
fig.update_yaxes(title_text='R²', row=1, col=2)
fig.update_yaxes(title_text='Reversion ratio', row=2, col=1)
fig.update_yaxes(title_text='gamma', row=2, col=2)
fig.update_xaxes(title_text='Total generated messages', row=2, col=2)

fig.update_layout(
    title='Recommendations Overview',
    width=1200, height=800,
    template='plotly_white',
    barmode='group',
    legend=dict(x=1.02, y=1, bgcolor='rgba(255,255,255,0.8)'),
)
fig.show()